# Clase 3. Procesos de Decisión de Markov (PDM)

**Duración estimada:** 105 minutos
**Capítulo del manual:** [04_procesos_decision_markov.md](../manual/04_procesos_decision_markov.md)

## De la máquina tragaperras al mapa completo

En la clase anterior conociste los bandidos: un agente que elige entre varias palancas y solo debe equilibrar exploración y explotación, porque cada tirada es independiente de las anteriores. Hoy das un paso más.

La vida real casi nunca funciona sin memoria. Cuando conduces, riegas un cultivo o gestionas el inventario de un almacén, cada decisión te deja en una situación distinta que condiciona qué opciones tienen sentido después. Necesitas un lenguaje formal que capture esa cadena de causa y efecto: el **Proceso de Decisión de Markov** (PDM), conocido en inglés como *Markov Decision Process* (MDP) —encontrarás ambos términos en el código y en la literatura—.

Al final de la clase tendrás en Python un mundo-cuadrícula (GridWorld) funcional que la próxima clase ampliará para encontrar la mejor estrategia posible.

### Objetivos de hoy

- Identificar los cinco elementos de un PDM: S, A, P, R y γ
- Entender la propiedad de Markov y por qué simplifica el problema
- Distinguir una política determinista de una estocástica
- Calcular Vπ de una política fija mediante evaluación iterativa

> 🎙️ **Guion de clase:** abre preguntando "¿qué decisión tomasteis hoy que dependía de una decisión anterior?". Deja que dos o tres personas respondan antes de introducir el PDM: sirve para anclar la idea de que el estado importa.

## 1. Los cinco elementos de un PDM

Piensa en un robot que recoge latas de aluminio para reciclar y funciona con batería. En cada instante se encuentra en uno de dos **estados** posibles según su nivel de carga: alto o bajo. Según el estado en el que esté, puede elegir entre varias **acciones**: esperar sin moverse, salir a buscar latas o, si va escaso de energía, volver a la base a recargar.

Aquí tienes los cinco ingredientes del PDM:

- **S** (states, estados): el conjunto de situaciones posibles en las que puede estar el agente —para el robot, {carga alta, carga baja}—
- **A** (actions, acciones): las opciones disponibles en cada estado —esperar, buscar o recargar—
- **P**: la función de transición, la probabilidad de terminar en cada estado siguiente según dónde estabas y qué acción tomaste. Buscar con la batería alta casi siempre te deja igual de cargado, pero existe una probabilidad de quedarte sin energía
- **R**: la función de recompensa, el número que recibes tras cada transición. Buscar latas da una recompensa positiva; quedarte tirado sin batería, una penalización fuerte
- **γ** (gamma): el factor de descuento, un número entre cero y uno que decide cuánto te importa el futuro

Un PDM completo es la tupla **(S, A, P, R, γ)**.

> 🎙️ **Guion de clase:** dibuja en la pizarra los dos estados del robot y las flechas de transición mientras lo explicas. Es la imagen mental que van a reutilizar durante toda la clase.

## 2. La propiedad de Markov: solo importa el presente

El nombre "Markov" no es decorativo. El PDM asume la **propiedad de Markov**: el futuro depende únicamente del estado actual, no de cómo llegaste hasta él. Da igual si el robot lleva dos horas buscando latas o si acaba de recargar: lo único relevante para decidir qué hacer ahora es su nivel de batería en este momento.

Esta suposición simplifica enormemente el problema: no necesitas guardar todo el historial de decisiones pasadas, basta con un estado bien diseñado que resuma lo importante. Por eso, al diseñar un sistema real —por ejemplo, el estado de un vehículo autónomo, que podría incluir posición, velocidad y obstáculos cercanos—, tu tarea es elegir una representación que se acerque lo máximo posible a esta propiedad.

## 3. La política: tu estrategia de decisión

Una **política**, escrita como π (pi), es la estrategia que sigue el agente: una regla que, para cada estado, indica qué acción tomar. Puede ser:

- **Determinista**: en el estado "batería baja", π siempre elige "recargar"
- **Estocástica**: en el estado "batería alta", π elige "buscar" el 70% de las veces y "esperar" el 30% restante

Una política aleatoria, que reparte la misma probabilidad entre todas las acciones disponibles, es el punto de partida más simple para razonar sobre un PDM, y es justo la que vas a programar hoy.

> 🎙️ **Guion de clase:** pregunta a la clase si una política estocástica les parece "peor" que una determinista. Aprovecha para adelantar que en RL a veces conviene mantener algo de aleatoriedad (conexión con la exploración de la clase de bandidos).

## 4. Cuánto vale un estado: la función de valor

Aquí llega la pregunta central de hoy: si sigues una política π, ¿cuánto "vale" estar en un estado concreto? La respuesta no es solo lo que ganas en este instante, sino la suma de todas las recompensas futuras, descontadas por γ. A esa suma se le llama **retorno** (Gt):

```
Gt = Rt+1 + γ·Rt+2 + γ²·Rt+3 + ...
```

El factor γ hace dos cosas a la vez: refleja que sueles preferir ganar algo hoy antes que la misma recompensa dentro de diez pasos, y evita que la suma crezca sin límite en tareas que podrían continuar indefinidamente, porque cada recompensa futura pesa un poco menos que la anterior. Un γ cercano a cero produce un agente miope, que solo mira el paso inmediato; uno cercano a uno lo vuelve previsor, dispuesto a sacrificar recompensa inmediata por un beneficio mayor más adelante.

A partir del retorno se definen dos **funciones de valor**:

- **Vπ(s)**: el retorno esperado si empiezas en el estado s y sigues la política π a partir de ahí
- **Qπ(s,a)**: el retorno esperado si, estando en el estado s, tomas la acción a y después sigues π

Ambas están relacionadas de forma directa: Vπ(s) es el promedio de Qπ(s,a) sobre todas las acciones, ponderado por la probabilidad que la política asigna a cada una.

## 5. Las ecuaciones de Bellman

El retorno se puede escribir de forma recursiva, porque Gt = Rt+1 + γ·Gt+1. En palabras: lo que vale este momento es la recompensa de este paso más el valor descontado de todo lo que viene después. Esa idea, aplicada a la función de valor, es la **ecuación de Bellman de expectación**:

```
Vπ(s) = Σ_a π(a|s) · Σ_s' P(s'|s,a) · [R(s,a,s') + γ·Vπ(s')]
```

No dejes que la notación te asuste: es solo una suma ponderada. Para cada acción posible (según tu política) y cada posible estado siguiente (según la probabilidad de transición), sumas la recompensa inmediata más su valor descontado. La misma idea, aplicada a Q, da la versión para pares estado-acción:

```
Qπ(s,a) = Σ_s' P(s'|s,a) · [R(s,a,s') + γ·Σ_a' π(a'|s') Qπ(s',a')]
```

Estas dos ecuaciones definen el valor de una política ya fija: son un sistema lineal, y por eso puedes resolverlas —o aproximarlas iterando, como vas a hacer en el código de hoy—.

> 🎙️ **Guion de clase:** insiste en la lectura verbal de la fórmula ("recompensa inmediata + valor descontado del estado siguiente") antes de tocar el teclado. Es la frase que van a repetir dentro de un momento en cada línea de código.

### Adelanto: la pregunta que abre la próxima clase

Todo lo anterior asume que ya tienes una política π y solo quieres saber cuánto vale. Pero la pregunta que de verdad interesa es otra: ¿cuál es la mejor política posible? A esa política óptima se le llama **π\***, y su función de valor **V\*(s)** cumple la **ecuación de optimalidad de Bellman**:

```
V*(s) = max_a Σ_s' P(s'|s,a) · [R(s,a,s') + γ·V*(s')]
```

La diferencia con la ecuación de expectación es sutil pero decisiva: en lugar de promediar sobre las acciones según tu política, te quedas con el máximo. Ese máximo convierte el sistema en no lineal y necesita métodos iterativos, que es justo de lo que trata la Clase 4: programación dinámica. Hoy te quedas en la mitad del camino: sabrás medir una política, no todavía mejorarla.

## 6. Manos a la obra: el GridWorld

Vas a construir un mundo-cuadrícula pequeño, de cuatro filas por cuatro columnas (4×4), con una única casilla terminal (la meta) en la esquina inferior derecha. El agente puede moverse en las cuatro direcciones cardinales; si intenta salir del tablero, se queda donde está. Cada paso cuesta −1 de recompensa, salvo llegar a la meta, donde el episodio termina.

Vas a definir una política aleatoria y calcularás su Vπ aplicando la ecuación de Bellman de expectación una y otra vez, hasta que los valores dejen de cambiar. Este proceso se llama **evaluación iterativa de políticas**.

Fíjate en que la clase `GridWorld` y las funciones se escriben para ser reutilizadas: la Clase 4 añadirá mejora de políticas encima de esta misma base, sin reescribirla. Construye el código célula a célula, ejecutando cada una antes de pasar a la siguiente.

> 🎙️ **Guion de clase:** antes de escribir código, dibuja la cuadrícula 4x4 en la pizarra con la meta marcada. Numera un par de casillas para que la clase visualice las coordenadas (fila, columna) antes de verlas en Python.

In [ ]:
# Solo necesitas numpy, ya viene preinstalado en Google Colab
import numpy as np

### Las acciones y sus desplazamientos

Cada acción mueve al agente una casilla en una dirección. Lo representamos como un delta de fila y un delta de columna.

In [ ]:
# 🎙️ Guion: pregunta "¿por qué represento 'arriba' como (-1, 0) y no como (0, -1)?"
# Respuesta: la primera coordenada es la fila, y subir una fila resta uno al índice.
ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]

DESPLAZAMIENTOS = {
    "arriba": (-1, 0),
    "abajo": (1, 0),
    "izquierda": (0, -1),
    "derecha": (0, 1),
}

### La clase `GridWorld`

Esta clase encapsula el entorno completo: qué estados existen, cómo se mueve el agente y qué recompensa recibe. Se diseña para que la Clase 4 la reutilice tal cual y le añada iteración de políticas e iteración de valores encima, así que mantén sus nombres exactos si luego la modificas.

In [ ]:
class GridWorld:
    """
    Un mundo-cuadrícula de n_filas x n_columnas con una casilla terminal (la meta).

    - Los estados son tuplas (fila, columna).
    - Las acciones son las cuatro direcciones cardinales.
    - Las transiciones son deterministas: moverse contra un muro o el
      borde del tablero deja al agente en la misma casilla.
    - La recompensa es 'recompensa_paso' en cada movimiento (por defecto -1),
      salvo en la casilla terminal, donde el episodio ya ha terminado.
    """

    def __init__(self, n_filas=4, n_columnas=4, estados_terminales=None, recompensa_paso=-1.0):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.recompensa_paso = recompensa_paso

        # Por defecto, la meta es la esquina inferior derecha
        if estados_terminales is None:
            estados_terminales = [(n_filas - 1, n_columnas - 1)]
        self.estados_terminales = set(estados_terminales)

        # Lista de todos los estados (fila, columna)
        self.estados = [(f, c) for f in range(n_filas) for c in range(n_columnas)]
        self.acciones = ACCIONES

    def es_terminal(self, estado):
        return estado in self.estados_terminales

    def estado_siguiente(self, estado, accion):
        """Devuelve la casilla resultante de aplicar 'accion' desde 'estado'.
        Si el movimiento choca con el borde del tablero, el agente se queda quieto."""
        if self.es_terminal(estado):
            return estado

        df, dc = DESPLAZAMIENTOS[accion]
        fila, columna = estado
        nueva_fila = max(0, min(self.n_filas - 1, fila + df))
        nueva_columna = max(0, min(self.n_columnas - 1, columna + dc))
        return (nueva_fila, nueva_columna)

    def recompensa(self, estado, accion, estado_siguiente):
        """Recompensa inmediata de la transición. En este GridWorld es fija
        (-1 por paso), pero se recibe (s, a, s') completo para poder
        ampliar el ejemplo más adelante (por ejemplo, recompensas distintas por casilla)."""
        if self.es_terminal(estado):
            return 0.0
        return self.recompensa_paso

### Comprobación rápida

Antes de seguir, instancia un `GridWorld` y comprueba que se comporta como esperas: que la esquina inferior derecha es terminal y que moverse contra el borde te deja en el mismo sitio.

> 🎙️ **Guion de clase:** pide a alguien de la clase que prediga en voz alta el resultado de `estado_siguiente((0, 0), "arriba")` antes de ejecutar la celda.

In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)

print("¿(3, 3) es terminal?", mundo.es_terminal((3, 3)))
print("Moverse 'arriba' desde (0, 0):", mundo.estado_siguiente((0, 0), "arriba"))
print("Moverse 'derecha' desde (1, 1):", mundo.estado_siguiente((1, 1), "derecha"))

### La política aleatoria

Representamos una política como un diccionario anidado: `politica[estado][accion]` es la probabilidad de tomar esa acción en ese estado. La política aleatoria reparte la misma probabilidad entre las cuatro direcciones.

In [ ]:
def politica_aleatoria(mundo):
    """Construye una política estocástica que, en cada estado, elige
    cualquiera de las cuatro acciones con la misma probabilidad (0.25).
    Se representa como un diccionario: politica[estado][accion] = probabilidad."""
    probabilidad_uniforme = 1.0 / len(mundo.acciones)
    return {
        estado: {accion: probabilidad_uniforme for accion in mundo.acciones}
        for estado in mundo.estados
    }

politica = politica_aleatoria(mundo)

# Inspecciona la política en un único estado
print("Política en el estado (2, 2):", politica[(2, 2)])

### Evaluación iterativa de políticas

Aquí está el corazón de la clase de hoy: aplicar la ecuación de Bellman de expectación repetidamente, barrido tras barrido, hasta que Vπ deje de cambiar (por debajo de un `umbral`).

Fíjate en la correspondencia línea a línea con la fórmula: `prob_accion * (r + gamma * V[estado_sig])` es exactamente `π(a|s) · [R(s,a,s') + γ·Vπ(s')]`. Como este GridWorld es determinista, P(s'|s,a) vale uno para el único `estado_siguiente` que devuelve `mundo.estado_siguiente()`, así que la suma sobre s' desaparece y queda un único término.

> 🎙️ **Guion de clase:** antes de ejecutar, señala en la pizarra qué línea de código corresponde a cada símbolo de la fórmula (π(a|s), R(s,a,s'), γ·Vπ(s')). Es el momento de mayor densidad conceptual de la clase, tómate tu tiempo.

In [ ]:
def evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """
    Calcula Vpi(s) para una política fija aplicando repetidamente
    la ecuación de Bellman de expectación:

        Vpi(s) = suma_a pi(a|s) * suma_s' P(s'|s,a) * [R(s,a,s') + gamma * Vpi(s')]

    Devuelve V como un diccionario {estado: valor} y el número de barridos usados.
    """
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)  # barrido síncrono: usamos siempre los valores del barrido anterior

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion

### Visualizar el resultado

Una función auxiliar para imprimir V como una cuadrícula legible, en vez de un diccionario plano.

In [ ]:
def imprimir_valores(mundo, V):
    """Muestra V como una cuadrícula legible, para inspeccionar el resultado."""
    for f in range(mundo.n_filas):
        fila_textos = []
        for c in range(mundo.n_columnas):
            fila_textos.append(f"{V[(f, c)]:6.2f}")
        print(" ".join(fila_textos))

### Ejecuta la evaluación completa

Junta todas las piezas: crea el mundo, la política aleatoria, evalúa Vπ e imprime el resultado.

> 🎙️ **Guion de clase:** antes de ejecutar, pregunta "¿en qué casilla esperáis el valor más alto (menos negativo)?". Confirma la intuición con el resultado.

In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)
politica = politica_aleatoria(mundo)

V, num_barridos = evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4)

print(f"Convergió en {num_barridos} barridos.\n")
print("Función de valor Vpi bajo la política aleatoria:")
imprimir_valores(mundo, V)

### Cómo leer el resultado

Deberías obtener algo parecido a esto (los valores exactos dependen de γ y del umbral de convergencia):

```
Convergió en 76 barridos.

Función de valor Vpi bajo la política aleatoria:
 -9.36  -9.22  -8.97  -8.75
 -9.22  -8.97  -8.50  -7.97
 -8.97  -8.50  -7.42  -5.76
 -8.75  -7.97  -5.76   0.00
```

Observa el patrón: las casillas más cercanas a la meta (la esquina inferior derecha, en 0.00) tienen un valor menos negativo, porque bajo una política aleatoria hacen falta en promedio menos pasos para llegar. Las más alejadas, como la esquina superior izquierda, arrastran más pasos de recompensa −1 antes de terminar. Eso es exactamente lo que mide Vπ: el coste esperado, descontado, de llegar hasta el final siguiendo tu política actual.

> 🎙️ **Guion de clase:** deja que la clase compare el patrón de números con el dibujo de la pizarra. Pregunta: "¿por qué la esquina superior izquierda es la peor casilla posible aquí?".

## Puntos clave

- Un PDM se define por la tupla (S, A, P, R, γ): estados, acciones, transiciones, recompensas y factor de descuento
- La propiedad de Markov dice que el futuro solo depende del estado presente, no del historial completo
- Una política π puede ser determinista o estocástica, y asigna a cada estado una acción o una distribución de probabilidad sobre ellas
- Vπ(s) mide el retorno esperado desde un estado siguiendo π; Qπ(s,a) hace lo mismo pero fijando también la primera acción
- La ecuación de Bellman de expectación descompone tu valor actual en recompensa inmediata más lo que vale, descontado, el estado siguiente
- La ecuación de optimalidad de Bellman sustituye el promedio ponderado por un máximo sobre acciones, y define V* y la política óptima π*
- La evaluación iterativa de políticas aplica la ecuación de Bellman de expectación repetidamente hasta que Vπ converge

## Cierre y próximos pasos

Ya tienes el mapa —el PDM completo— y sabes calcular cuánto vale seguir una política concreta. En la Clase 4 vas a usar este mismo `GridWorld` para dar el siguiente paso: en lugar de solo evaluar una estrategia fija, la mejorarás paso a paso mediante iteración de políticas e iteración de valores, hasta llegar a la política óptima π*.

**Antes de la próxima clase:**
- **Abre** el notebook de práctica `clase03_practica_procesos_decision_markov.ipynb` y completa los dos ejercicios
- **Experimenta** cambiando γ a 0.5 y a 0.99: anota cómo cambian los valores y el número de barridos hasta converger
- **Repasa** el capítulo 4 del manual si quieres profundizar en la derivación matemática

> 🎙️ **Guion de clase:** cierra recordando la frase clave de hoy: "recompensa inmediata + valor descontado del estado siguiente". Es la misma frase que van a repetir la próxima clase, ahora con un máximo en vez de un promedio.